# ShExMap Validation & Materialisation

This notebook demonstrates the full ShExMap workflow:
1. **Query** the QLever triplestore for a pairing — retrieves both ShEx schemas, sample Turtle data, and focus nodes.
2. **Validate** the source RDF against the source ShEx and extract Map bindings.
3. **Materialise** the target RDF by applying the bindings to the target ShEx.
4. **Pretty-print** the resulting Turtle.

Requires the ShExMap Repository stack to be running (`docker compose up`).

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
SPARQL_ENDPOINT = "http://localhost:7001/sparql"   # QLever direct
API_BASE        = "http://localhost:8080/api/v1"   # via nginx (host-mapped port)

import json, textwrap, requests
from pprint import pprint

## 1 — Query the triplestore for a pairing

We ask for all pairings and, for each, retrieve:
- pairing title, source/target focus IRIs
- source & target ShEx content (from the current version node)
- source & target sample Turtle data

In [ ]:
SPARQL_QUERY = """
PREFIX dct:      <http://purl.org/dc/terms/>
PREFIX shexmap:  <https://w3id.org/shexmap/ontology#>
PREFIX schema:   <https://schema.org/>

SELECT
  ?pairingTitle
  ?srcTitle  ?srcShEx  ?srcTurtle  ?srcFocus
  ?tgtTitle  ?tgtShEx  ?tgtTurtle  ?tgtFocus
WHERE {
  # ── Pairing ────────────────────────────────────────────────────────────────
  ?pairing a shexmap:ShExMapPairing ;
           dct:title      ?pairingTitle ;
           shexmap:sourceMap ?srcMap ;
           shexmap:targetMap ?tgtMap .

  OPTIONAL { ?pairing shexmap:sourceFocusIri ?srcFocus }
  OPTIONAL { ?pairing shexmap:targetFocusIri ?tgtFocus }

  # ── Source map ─────────────────────────────────────────────────────────────
  ?srcMap dct:title ?srcTitle .
  OPTIONAL { ?srcMap shexmap:sampleTurtleData ?srcTurtle }

  # current version → ShEx content
  OPTIONAL {
    ?srcMap shexmap:currentVersion ?srcVer .
    ?srcVer shexmap:versionContent ?srcShEx .
  }

  # ── Target map ─────────────────────────────────────────────────────────────
  ?tgtMap dct:title ?tgtTitle .
  OPTIONAL { ?tgtMap shexmap:sampleTurtleData ?tgtTurtle }

  OPTIONAL {
    ?tgtMap shexmap:currentVersion ?tgtVer .
    ?tgtVer shexmap:versionContent ?tgtShEx .
  }
}
"""

resp = requests.get(
    SPARQL_ENDPOINT,
    params={"query": SPARQL_QUERY, "format": "json"},
    timeout=30,
)
resp.raise_for_status()
results = resp.json()["results"]["bindings"]

print(f"Found {len(results)} pairing(s)\n")
for r in results:
    print(f"  Pairing : {r['pairingTitle']['value']}")
    print(f"  Source  : {r['srcTitle']['value']}")
    print(f"  Target  : {r['tgtTitle']['value']}")
    print()

## 2 — Select a pairing and inspect its components

In [ ]:
# Use the first pairing found
p = results[0]

pairing_title = p["pairingTitle"]["value"]
src_title     = p["srcTitle"]["value"]
tgt_title     = p["tgtTitle"]["value"]
src_shex      = p.get("srcShEx",   {}).get("value", "")
tgt_shex      = p.get("tgtShEx",   {}).get("value", "")
src_turtle    = p.get("srcTurtle", {}).get("value", "")
tgt_turtle    = p.get("tgtTurtle", {}).get("value", "")
src_focus     = p.get("srcFocus",  {}).get("value", "")
tgt_focus     = p.get("tgtFocus",  {}).get("value", "")

print(f"=== Pairing: {pairing_title} ===")
print()
print(f"--- Source ShEx ({src_title}) ---")
print(src_shex)
print(f"--- Source Turtle ---")
print(src_turtle)
print(f"--- Source Focus IRI ---")
print(src_focus)
print()
print(f"--- Target ShEx ({tgt_title}) ---")
print(tgt_shex)
print(f"--- Target Turtle (sample, for reference) ---")
print(tgt_turtle)
print(f"--- Target Focus IRI ---")
print(tgt_focus)

## 3 — Validate source RDF and materialise the target

We `POST` to `/api/v1/validate` with:
- `sourceShEx` + `sourceRdf` + `sourceNode` → validates & extracts Map variable bindings
- `targetShEx` + `targetNode` → materialises new RDF using those bindings

In [ ]:
payload = {
    "sourceShEx":  src_shex,
    "sourceRdf":   src_turtle,
    "sourceNode":  src_focus,
    "targetShEx":  tgt_shex,
    "targetNode":  tgt_focus,
}

resp = requests.post(
    f"{API_BASE}/validate",
    json=payload,
    timeout=30,
)
resp.raise_for_status()
result = resp.json()

print(f"valid   : {result['valid']}")
if result.get("errors"):
    print(f"errors  : {result['errors']}")

## 4 — Map variable bindings

In [ ]:
bindings = result.get("bindings", {})
print(f"{len(bindings)} binding(s) extracted:\n")
for var, val in sorted(bindings.items()):
    short_var = var.split("#")[-1] if "#" in var else var.split("/")[-1]
    print(f"  {short_var:25s} = {val}")

## 5 — Pretty-print the materialised Turtle

The API returns Turtle as a plain string. We pretty-print it using `rdflib` if available, otherwise fall back to simple indented output.

In [ ]:
target_rdf = result.get("targetRdf", "")

if not target_rdf:
    print("No target RDF produced.")
else:
    # Try rdflib for canonical pretty-printing; fall back to raw text.
    try:
        import rdflib
        g = rdflib.Graph()
        g.parse(data=target_rdf, format="turtle")
        pretty = g.serialize(format="turtle")
        print("=== Materialised Turtle (pretty-printed via rdflib) ===")
        print(pretty)
    except ImportError:
        # rdflib not installed — print the raw N3Writer output with mild formatting
        import re
        lines = []
        for line in target_rdf.splitlines():
            line = line.rstrip()
            if line:
                lines.append(line)
        print("=== Materialised Turtle ===")
        print("\n".join(lines))

## 6 — Binding tree (structured view)

In [ ]:
def print_tree(node, indent=0):
    pad = "  " * indent
    print(f"{pad}[{node['shape']}]  focus: {node['focus']}")
    for b in node.get("bindings", []):
        short = b['variable'].split('#')[-1] if '#' in b['variable'] else b['variable'].split('/')[-1]
        print(f"{pad}  {short} = {b['value']}")
    for child in node.get("children", []):
        print_tree(child, indent + 1)

print("=== Binding tree ===")
for root in result.get("bindingTree", []):
    print_tree(root)